# Track A — Segmentation: Indian Bank Transactions

**Track:** A — RFM behavioral clustering 
**Data:** `data/processed/track_a_features_model.csv` (or raw fallback)
**Methods:** K-Means · GMM · Hierarchical · Autoencoder+KMeans
**Output:** `artifacts/experiments/comparison_board_track_a.csv`

Run cells top-to-bottom. Data loading auto-falls-back to raw if processed files are absent.


In [ ]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
K_RANGE = range(3, 9)
print('=== IMPORTS OK — Track A: Indian Bank ===')

## 1. Data Loading


In [ ]:
# ── Try pipeline-processed files first; fall back to raw data FE ──────────
processed_model  = Path('../data/processed/track_a_features_model.csv')
processed_interp = Path('../data/processed/track_a_features_interpretable.csv')

if processed_model.exists() and processed_interp.exists():
    df_model  = pd.read_csv(processed_model)
    df_interp = pd.read_csv(processed_interp)
    print(f'Loaded from pipeline outputs.')
    print(f'  model  : {df_model.shape}')
    print(f'  interp : {df_interp.shape}')
else:
    print('Pipeline outputs not found — running inline FE from raw data...')
    raw_candidates = [
        Path('../data/bank_transactions.csv'),
        Path('../data/raw/bank_transactions.csv'),
    ]
    raw_path = next((p for p in raw_candidates if p.exists()), None)
    assert raw_path, 'Raw data not found. Check data/bank_transactions.csv or data/raw/bank_transactions.csv'

    df_raw = pd.read_csv(raw_path).dropna()
    print(f'  Raw loaded: {df_raw.shape}')

    # Age (century-pivot fix for 2-digit years)
    df_raw['DOB'] = pd.to_datetime(df_raw['CustomerDOB'], format='%d/%m/%y', errors='coerce')
    df_raw.loc[df_raw['DOB'].dt.year > 2005, 'DOB'] -= pd.DateOffset(years=100)
    df_raw['Age'] = (2016 - df_raw['DOB'].dt.year)
    df_raw['Age'] = df_raw['Age'].where(df_raw['Age'].between(18, 100))

    # Gender one-hot
    df_raw = pd.get_dummies(df_raw, columns=['CustGender'], drop_first=True, dtype=int)
    gender_col = 'CustGender_M' if 'CustGender_M' in df_raw.columns else None

    # Location tier
    TIER1 = {'MUMBAI','DELHI','BANGALORE','CHENNAI','HYDERABAD','KOLKATA','PUNE','AHMEDABAD','NEW DELHI'}
    df_raw['Is_Tier_1_City'] = df_raw['CustLocation'].str.upper().isin(TIER1).astype(int)

    # Time features
    df_raw['TransactionDate'] = pd.to_datetime(df_raw['TransactionDate'], format='%d/%m/%y', errors='coerce')
    df_raw['Is_Weekend'] = df_raw['TransactionDate'].dt.dayofweek.isin([5, 6]).astype(int)
    if 'TransactionTime' in df_raw.columns:
        df_raw['HourOfDay'] = df_raw['TransactionTime'] // 10000
        df_raw['Is_Night']   = ((df_raw['HourOfDay'] >= 20) | (df_raw['HourOfDay'] < 4)).astype(int)
    else:
        df_raw['Is_Night'] = 0

    df_raw['Spend_Balance_Ratio'] = (
        df_raw['TransactionAmount (INR)'] / (df_raw['CustAccountBalance'] + 1)
    )
    max_date = df_raw['TransactionDate'].max()

    # Aggregate to customer grain
    agg = dict(
        Age=('Age', 'first'),
        AccountBalance=('CustAccountBalance', 'mean'),
        Last_Transaction=('TransactionDate', 'max'),
        Transaction_Count=('TransactionID', 'count'),
        Avg_Transaction_Amt=('TransactionAmount (INR)', 'mean'),
        Std_Transaction_Amt=('TransactionAmount (INR)', 'std'),
        Total_Spend=('TransactionAmount (INR)', 'sum'),
        Avg_Spend_Ratio=('Spend_Balance_Ratio', 'mean'),
        Weekend_Spend_Ratio=('Is_Weekend', 'mean'),
        Night_Spend_Ratio=('Is_Night', 'mean'),
        Is_Tier_1_City=('Is_Tier_1_City', 'first'),
    )
    if gender_col:
        agg['Gender_M'] = (gender_col, 'first')

    df_cust = df_raw.groupby('CustomerID').agg(**agg).reset_index()
    df_cust['Std_Transaction_Amt'] = df_cust['Std_Transaction_Amt'].fillna(0)
    df_cust['Recency_Days'] = (max_date - df_cust['Last_Transaction']).dt.days
    df_cust.drop(columns=['Last_Transaction'], inplace=True, errors='ignore')
    df_cust.dropna(inplace=True)

    # Build model matrix (log1p monetary cols + StandardScaler)
    LOG_COLS = ['AccountBalance', 'Avg_Transaction_Amt', 'Std_Transaction_Amt', 'Total_Spend']
    df_interp = df_cust.copy()
    df_model  = df_cust.copy()
    for c in LOG_COLS:
        if c in df_model.columns:
            df_model[c] = np.log1p(df_model[c])

    id_col   = 'CustomerID'
    num_cols = [c for c in df_model.columns if c != id_col
                and pd.api.types.is_numeric_dtype(df_model[c])]
    scaler = StandardScaler()
    df_model[num_cols] = scaler.fit_transform(df_model[num_cols])
    print(f'  Inline FE complete: {df_model.shape}')

print(f'\nCustomer universe : {len(df_model):,}')
print(f'Feature columns   : {[c for c in df_model.columns if c != "CustomerID"]}')

In [ ]:
ID_COL   = 'CustomerID' if 'CustomerID' in df_model.columns else df_model.columns[0]
FEAT_COLS = [c for c in df_model.columns
             if c != ID_COL and pd.api.types.is_numeric_dtype(df_model[c])]

X_full = df_model[FEAT_COLS].fillna(0).values.astype(float)

# Sample for silhouette / hierarchical (O(n^2) — cap at 50k)
SAMPLE_N  = 50_000
sample_idx = np.random.choice(len(X_full), min(SAMPLE_N, len(X_full)), replace=False)
X_sample  = X_full[sample_idx]

print(f'Full feature matrix : {X_full.shape}')
print(f'Stability sample    : {X_sample.shape}')
print(f'Features            : {FEAT_COLS}')

## 2. K-Means Sweep (k = 3–8)


In [ ]:
kmeans_results = {}
inertia, sil_k, db_k, ch_k = [], [], [], []
SIL_SAMPLE = 10_000

print('=== K-MEANS SWEEP ===')
for k in K_RANGE:
    km  = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X_full)
    idx = np.random.choice(len(X_full), min(SIL_SAMPLE, len(X_full)), replace=False)
    sil = silhouette_score(X_full[idx], lbl[idx])
    db  = davies_bouldin_score(X_full[idx], lbl[idx])
    ch  = calinski_harabasz_score(X_full[idx], lbl[idx])
    inertia.append(km.inertia_); sil_k.append(sil); db_k.append(db); ch_k.append(ch)
    kmeans_results[k] = dict(model=km, labels=lbl, sil=sil, db=db, ch=ch)
    sizes = pd.Series(lbl).value_counts().sort_index().to_dict()
    print(f'  k={k} | sil={sil:.3f} | DB={db:.3f} | CH={ch:.0f} | sizes={sizes}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ks = list(K_RANGE)
axes[0].plot(ks, inertia, 'b-o');   axes[0].set_title('Elbow (Inertia)');     axes[0].set_xlabel('k')
axes[1].plot(ks, sil_k,  'g-o');   axes[1].set_title('Silhouette Score');    axes[1].set_xlabel('k')
axes[2].plot(ks, db_k,   'r-o');   axes[2].set_title('Davies-Bouldin Index');axes[2].set_xlabel('k')
plt.suptitle('Track A — K-Means Sweep', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

best_k_km = ks[int(np.argmax(sil_k))]
print(f'Best k (silhouette): {best_k_km}')

## 3. Gaussian Mixture Model Sweep (k = 3–8)


In [ ]:
gmm_results = {}
gmm_bic, gmm_sil = [], []
GMM_FIT_N = 50_000
idx_gmm   = np.random.choice(len(X_full), min(GMM_FIT_N, len(X_full)), replace=False)
X_gmm     = X_full[idx_gmm]

print('=== GMM SWEEP ===')
for k in K_RANGE:
    gmm = GaussianMixture(n_components=k, n_init=3, covariance_type='full', random_state=RANDOM_STATE)
    gmm.fit(X_gmm)
    lbl  = gmm.predict(X_gmm)
    prob = gmm.predict_proba(X_gmm).max(axis=1)
    sil  = silhouette_score(X_gmm, lbl)
    db   = davies_bouldin_score(X_gmm, lbl)
    ch   = calinski_harabasz_score(X_gmm, lbl)
    gmm_bic.append(gmm.bic(X_gmm))
    gmm_sil.append(sil)
    gmm_results[k] = dict(model=gmm, labels=lbl, confidence=prob, sil=sil, db=db, ch=ch)
    print(f'  k={k} | sil={sil:.3f} | DB={db:.3f} | BIC={gmm_bic[-1]:.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_RANGE), gmm_bic, 'b-o'); axes[0].set_title('GMM BIC'); axes[0].set_xlabel('k')
axes[1].plot(list(K_RANGE), gmm_sil, 'g-o'); axes[1].set_title('GMM Silhouette'); axes[1].set_xlabel('k')
plt.suptitle('Track A — GMM Sweep', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

best_k_gmm = list(K_RANGE)[int(np.argmax(gmm_sil))]
print(f'Best k (GMM silhouette): {best_k_gmm}')

## 4. Hierarchical Clustering + Dendrogram


In [ ]:
print('=== HIERARCHICAL CLUSTERING ===')
DEND_N = 2000
Z_dend = linkage(X_sample[:DEND_N], method='ward')

fig, ax = plt.subplots(figsize=(13, 5))
dendrogram(Z_dend, ax=ax, truncate_mode='lastp', p=20,
           leaf_rotation=45, leaf_font_size=9)
ax.set_title('Track A — Hierarchical Dendrogram (Ward, sample of 2000)')
ax.set_xlabel('Cluster size'); ax.set_ylabel('Distance')
plt.tight_layout(); plt.show()

hier_results = {}
for k in [best_k_km, min(best_k_km + 1, 8)]:
    agg_clust = AgglomerativeClustering(n_clusters=k, linkage='ward')
    lbl = agg_clust.fit_predict(X_sample)
    sil = silhouette_score(X_sample, lbl)
    db  = davies_bouldin_score(X_sample, lbl)
    ch  = calinski_harabasz_score(X_sample, lbl)
    hier_results[k] = dict(labels=lbl, sil=sil, db=db, ch=ch)
    print(f'  k={k} | sil={sil:.3f} | DB={db:.3f} | CH={ch:.0f}')

best_k_hier = max(hier_results, key=lambda k: hier_results[k]['sil'])
print(f'Best k (hierarchical): {best_k_hier}')

## 5. Autoencoder + K-Means on Latent Space

> **Recommended flag:** `False` for Track A — K-Means is the natural fit.
> Running AE here anyway to let the rubric decide.


In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
    print(f'PyTorch {torch.__version__} available.')
except ImportError:
    TORCH_OK = False
    print('PyTorch not installed — AE track will be skipped.')
    print('  Install: pip install torch')

if TORCH_OK:
    class Autoencoder(nn.Module):
        def __init__(self, n_feat, lat_dim=16):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Linear(n_feat, 64), nn.ReLU(),
                nn.Linear(64, 32),     nn.ReLU(),
                nn.Linear(32, lat_dim)
            )
            self.dec = nn.Sequential(
                nn.Linear(lat_dim, 32), nn.ReLU(),
                nn.Linear(32, 64),      nn.ReLU(),
                nn.Linear(64, n_feat)
            )
        def forward(self, x):
            z = self.enc(x)
            return self.dec(z), z

    def train_ae(X_np, lat_dim=16, epochs=50, batch_sz=256, lr=1e-3):
        X_t    = torch.tensor(X_np, dtype=torch.float32)
        model  = Autoencoder(X_np.shape[1], lat_dim)
        opt    = torch.optim.Adam(model.parameters(), lr=lr)
        crit   = nn.MSELoss()
        loader = DataLoader(TensorDataset(X_t), batch_size=batch_sz, shuffle=True)
        losses = []
        for ep in range(epochs):
            model.train(); ep_loss = 0
            for (b,) in loader:
                opt.zero_grad()
                loss = crit(model(b)[0], b)
                loss.backward()
                opt.step()
                ep_loss += loss.item()
            losses.append(ep_loss / len(loader))
            if (ep + 1) % 10 == 0:
                print(f'  epoch {ep+1:3d}/{epochs} | loss {losses[-1]:.5f}')
        model.eval()
        with torch.no_grad(): _, z = model(X_t)
        return z.numpy(), losses

    print('Autoencoder class ready.')

In [ ]:
ae_results = {}

if TORCH_OK:
    AE_N = min(100_000, len(X_full))
    ae_idx = np.random.choice(len(X_full), AE_N, replace=False)
    X_ae   = X_full[ae_idx].astype(np.float32)

    for lat in [8, 16]:
        print(f'\n--- AE latent_dim={lat} ---')
        torch.manual_seed(RANDOM_STATE)
        z, losses = train_ae(X_ae, lat_dim=lat, epochs=50)

        km_z = KMeans(n_clusters=best_k_km, n_init=20, random_state=RANDOM_STATE)
        lbl  = km_z.fit_predict(z)
        sil  = silhouette_score(z, lbl)
        db   = davies_bouldin_score(z, lbl)
        ch   = calinski_harabasz_score(z, lbl)
        ae_results[lat] = dict(z=z, labels=lbl, losses=losses, sil=sil, db=db, ch=ch)
        print(f'  k={best_k_km} on z | sil={sil:.3f} | DB={db:.3f} | CH={ch:.0f}')

    # Plot reconstruction losses
    fig, ax = plt.subplots(figsize=(8, 4))
    for lat, r in ae_results.items():
        ax.plot(r['losses'], label=f'latent={lat}')
    ax.set_title('AE Reconstruction Loss (Track A)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('AE skipped (PyTorch not available).')

## 6. Stability Analysis (Multi-seed ARI)


In [ ]:
N_SEEDS   = 10
ari_scores = []
base_lbl  = KMeans(n_clusters=best_k_km, n_init=20, random_state=0).fit_predict(X_sample)

for seed in range(1, N_SEEDS + 1):
    lbl_s = KMeans(n_clusters=best_k_km, n_init=5, random_state=seed).fit_predict(X_sample)
    ari_scores.append(adjusted_rand_score(base_lbl, lbl_s))

mean_ari = float(np.mean(ari_scores))
std_ari  = float(np.std(ari_scores))
print(f'=== STABILITY (K-Means k={best_k_km}, {N_SEEDS} seeds) ===')
print(f'  ARI mean={mean_ari:.3f} | std={std_ari:.3f} | min={min(ari_scores):.3f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, N_SEEDS + 1), ari_scores, color='steelblue', alpha=0.8)
ax.axhline(mean_ari, color='red', ls='--', label=f'mean={mean_ari:.3f}')
ax.set_title(f'Multi-seed Stability — K-Means k={best_k_km}')
ax.set_xlabel('Seed'); ax.set_ylabel('ARI vs seed 0'); ax.set_ylim(0, 1.05)
ax.legend(); plt.tight_layout(); plt.show()

## 7. Comparison Board


In [ ]:
rows = []

# K-Means
for k, r in kmeans_results.items():
    rows.append(dict(run_id=f'a_kmeans_k{k}', track='indian_bank', method='kmeans',
                     k=k, latent_dim='-',
                     silhouette=round(r['sil'],4), db_index=round(r['db'],4),
                     ch_index=round(r['ch'],0),
                     stability_ari=round(mean_ari,3) if k==best_k_km else None,
                     recommended=(k==best_k_km)))

# GMM
for k, r in gmm_results.items():
    rows.append(dict(run_id=f'a_gmm_k{k}', track='indian_bank', method='gmm',
                     k=k, latent_dim='-',
                     silhouette=round(r['sil'],4), db_index=round(r['db'],4),
                     ch_index=round(r['ch'],0),
                     stability_ari=None, recommended=False))

# Hierarchical
for k, r in hier_results.items():
    rows.append(dict(run_id=f'a_hier_k{k}', track='indian_bank', method='hierarchical',
                     k=k, latent_dim='-',
                     silhouette=round(r['sil'],4), db_index=round(r['db'],4),
                     ch_index=round(r['ch'],0),
                     stability_ari=None, recommended=False))

# AE
for lat, r in ae_results.items():
    rows.append(dict(run_id=f'a_ae_ld{lat}_k{best_k_km}', track='indian_bank',
                     method='ae_kmeans', k=best_k_km, latent_dim=lat,
                     silhouette=round(r['sil'],4), db_index=round(r['db'],4),
                     ch_index=round(r['ch'],0),
                     stability_ari=None, recommended=False))

board_a = pd.DataFrame(rows).sort_values('silhouette', ascending=False).reset_index(drop=True)

out_dir = Path('../artifacts/experiments')
out_dir.mkdir(parents=True, exist_ok=True)
board_a.to_csv(out_dir / 'comparison_board_track_a.csv', index=False)

print('=== COMPARISON BOARD — Track A ===')
display(board_a)
print(f'Saved to {out_dir}/comparison_board_track_a.csv')

## 8. Visualizations


In [ ]:
k = best_k_km
labels_best = kmeans_results[k]['labels']

# ── Cluster sizes ─────────────────────────────────────────────────
seg_counts = pd.Series(labels_best).value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette('Set2', k)
axes[0].bar(seg_counts.index, seg_counts.values, color=palette)
for i, v in zip(seg_counts.index, seg_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)
axes[0].set_title(f'Track A — Segment Sizes (K-Means k={k})')
axes[0].set_xlabel('Segment'); axes[0].set_ylabel('Customers')

# ── PCA 2D scatter ────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_sample)
lbl_pca = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_sample)
axes[1].scatter(X_pca[:,0], X_pca[:,1], c=lbl_pca, cmap='Set2', alpha=0.3, s=3)
axes[1].set_title(f'PCA 2D — sample (k={k})')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.suptitle('Track A — K-Means Best Solution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Centroid heatmap ──────────────────────────────────────────────
centroids = kmeans_results[k]['model'].cluster_centers_
c_df = pd.DataFrame(centroids, columns=FEAT_COLS,
                    index=[f'Seg {i}' for i in range(k)])
fig, ax = plt.subplots(figsize=(max(10, len(FEAT_COLS)), k + 1))
sns.heatmap(c_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Track A — K-Means Centroid Heatmap (scaled features)')
plt.tight_layout(); plt.show()

## 9. Segment Profiles (Interpretable Scale)


In [ ]:
# Profile on interpretable scale
id_col_i = 'CustomerID' if 'CustomerID' in df_interp.columns else df_interp.columns[0]
profile_df = df_interp.copy().reset_index(drop=True)
n_lbl = min(len(labels_best), len(profile_df))
profile_df = profile_df.iloc[:n_lbl].copy()
profile_df['segment'] = labels_best[:n_lbl]

prof_cols = [c for c in df_interp.columns
             if c != id_col_i and pd.api.types.is_numeric_dtype(df_interp[c])][:10]

seg_profile = profile_df.groupby('segment')[prof_cols].mean().round(2)
print('=== SEGMENT PROFILES (interpretable scale) ===')
display(seg_profile)

print('\n=== NAMING SUGGESTIONS (data-driven, refine after analysis) ===')
for seg in range(k):
    row = seg_profile.loc[seg] if seg in seg_profile.index else {}
    bal   = row.get('AccountBalance', 0)
    spend = row.get('Avg_Transaction_Amt', row.get('Total_Spend', 0))
    rec   = row.get('Recency_Days', 0)
    print(f'  Seg {seg}: Bal≈{bal:>10,.0f} | AvgSpend≈{spend:>8,.0f} | Recency={rec:.0f}d')

## 10. Summary


In [ ]:
best_run = board_a.iloc[0]
print('=' * 60)
print('TRACK A — SEGMENTATION COMPLETE')
print('=' * 60)
print(f'  Best run   : {best_run["run_id"]}')
print(f'  Method     : {best_run["method"]}')
print(f'  k          : {best_run["k"]}')
print(f'  Silhouette : {best_run["silhouette"]}')
print(f'  DB Index   : {best_run["db_index"]}')
print(f'  Stability  : {best_run["stability_ari"]}')
print()
print('Recommended flag marks EDA prior (not post-hoc selection).')
print('Rubric scoring happens in the comparison board after all 3 tracks run.')
print('\nNext → 07_Segmentation_Track_B_Santander.ipynb')